# Results Overview — Career Extraction

Aggregates submitted results from `results/register.csv` for career history extraction.
Evaluation uses temporal fuzzy matching (±1 year tolerance) at the spell level.
Metrics: macro-averaged **precision**, **recall**, **F1** across all test cases.

## 1. Imports & config

In [ ]:
import json as _json
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"]   = "white"

REGISTER = Path("/home/tom/projects/corex_eval/results/register.csv")
TASK     = "extraction"
VAR      = "career"

FAMILY_COLORS = {
    "OSS LLM": "#2196F3",
    "Closed":  "#FF5722",
}

## 2. Model catalogue

In [ ]:
# (family, display_name, experiment_folder)
CATALOGUE = [
    ("OSS LLM", "Llama 3.3 70B",  "lmstudio_llama33_70b_career"),
    ("OSS LLM", "Qwen3 80B",       "lmstudio_qwen3_80b_career"),
    ("Closed",  "Claude Opus 4.6", "claude_opus46_career"),
    ("Closed",  "GPT-5.3",         "gpt53_career"),
]

## 3. Load register

In [ ]:
reg = pd.read_csv(REGISTER)
reg = reg[(reg["task"] == TASK) & (reg["variable"] == VAR)].copy()
print(f"Total career extraction rows: {len(reg)}")

def _folder(path):
    parts = str(path).replace("\\", "/").split("/")
    return parts[-2] if len(parts) >= 2 else str(path)

reg["exp_folder"] = reg["experiment_path"].apply(_folder)
for col in ["precision", "recall", "f1"]:
    reg[col] = pd.to_numeric(reg[col], errors="coerce")

reg_best = (
    reg[reg["f1"].notna() & (reg["f1"] > 0)]
    .sort_values("f1", ascending=False)
    .drop_duplicates(subset="exp_folder", keep="first")
    .set_index("exp_folder")
)
print(f"Valid rows: {len(reg_best)}")
print("Folders found:", sorted(reg_best.index.tolist()))

## 4. Results table

In [ ]:
records = []
for family, name, folder in CATALOGUE:
    if folder in reg_best.index:
        row = reg_best.loc[folder]
        records.append({
            "Family":    family,
            "Model":     name,
            "Precision": round(float(row["precision"]), 4) if pd.notna(row["precision"]) else None,
            "Recall":    round(float(row["recall"]),    4) if pd.notna(row["recall"])    else None,
            "F1":        round(float(row["f1"]),        4) if pd.notna(row["f1"])        else None,
            "N cases":   int(row["n_evaluated"])             if pd.notna(row["n_evaluated"]) else None,
        })
    else:
        records.append({"Family": family, "Model": name,
                        "Precision": None, "Recall": None, "F1": None, "N cases": None})

results_df = pd.DataFrame(records)

def fmt_table(df):
    d = df.drop(columns=["Family"]).copy()
    for col in ["Precision", "Recall", "F1"]:
        d[col] = d[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "TBD")
    return d.style.hide(axis="index")

fmt_table(results_df)

## 5. Bar chart — Precision / Recall / F1

In [ ]:
has = results_df[results_df["F1"].notna()].copy()
x   = np.arange(len(has))
w   = 0.26

fig, ax = plt.subplots(figsize=(max(7, len(has) * 1.4), 5))

def _rgba(hex_color, alpha):
    r, g, b = mcolors.to_rgb(hex_color)
    return (r, g, b, alpha)

for (metric, alpha), offset in zip([("Precision", 1.0), ("Recall", 0.6), ("F1", 0.35)], [-w, 0, w]):
    vals   = has[metric].values.astype(float)
    colors = [_rgba(FAMILY_COLORS[f], alpha) for f in has["Family"]]
    bars   = ax.bar(x + offset, vals, w, color=colors, edgecolor="white", label=metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(has["Model"].tolist(), fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Career extraction — Precision / Recall / F1 by model",
             fontsize=12, fontweight="bold")

family_patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
metric_patches = [mpatches.Patch(color="grey", alpha=a, label=m)
                  for m, a in [("Precision", 1.0), ("Recall", 0.6), ("F1", 0.35)]]
ax.legend(handles=family_patches + metric_patches, fontsize=8, ncol=2, loc="upper right")

plt.tight_layout()
plt.show()

## 6. Per-case F1 distribution

In [ ]:
# Load per_case breakdowns from metrics_json
MODEL_FOLDER = "gpt53_career"   # change to any folder

def get_per_case(folder):
    if folder not in reg_best.index:
        print(f"'{folder}' not in register")
        return {}
    try:
        return _json.loads(str(reg_best.loc[folder, "metrics_json"])).get("per_case", {})
    except Exception as e:
        print(f"JSON parse error: {e}")
        return {}

pc = get_per_case(MODEL_FOLDER)
if pc:
    pc_df = pd.DataFrame(pc).T.astype(float)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, metric in zip(axes, ["precision", "recall", "f1"]):
        ax.hist(pc_df[metric].dropna(), bins=20, color="#2196F3", edgecolor="white")
        ax.axvline(pc_df[metric].mean(), color="red", linestyle="--", label=f"mean={pc_df[metric].mean():.3f}")
        ax.set_title(metric.capitalize())
        ax.set_xlabel("Score")
        ax.legend(fontsize=8)
    fig.suptitle(f"Per-case distribution — {MODEL_FOLDER}", fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"\nSummary:\n{pc_df[['precision','recall','f1','n_gold','n_predicted','n_matched']].describe().round(3)}")

## 7. Compare per-case F1 across models

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

all_f1s = {}
for family, name, folder in CATALOGUE:
    pc = get_per_case(folder)
    if pc:
        f1s = [v["f1"] for v in pc.values() if "f1" in v]
        all_f1s[name] = f1s

if all_f1s:
    ax.boxplot(all_f1s.values(), labels=all_f1s.keys(), patch_artist=True,
               boxprops=dict(facecolor="#90CAF9"), medianprops=dict(color="red", linewidth=2))
    ax.set_ylabel("Per-case F1")
    ax.set_ylim(0, 1.05)
    ax.set_title("Career extraction — per-case F1 distribution by model", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 8. LaTeX table

In [ ]:
def to_latex(df, caption, label):
    cols = [c for c in df.columns if c != "Family"]
    header_str = " & ".join(c.replace("_", "\\_") for c in cols) + " \\\\"
    rows_str = []
    prev_family = None
    for _, row in df.iterrows():
        if row["Family"] != prev_family and prev_family is not None:
            rows_str.append("\\midrule")
        prev_family = row["Family"]
        rows_str.append(" & ".join(str(row[c]) if pd.notna(row[c]) else "TBD" for c in cols) + " \\\\")
    body = "\n    ".join(rows_str)
    n_cols = len(cols)
    return (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\small\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"\\begin{{tabular}}{{l{'c' * (n_cols - 1)}}}\n"
        "\\toprule\n"
        f"{header_str}\n"
        "\\midrule\n"
        f"    {body}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}"
    )

print(to_latex(results_df, "Career extraction results", "tab:extraction-career"))